In [0]:
!pip install laspy[lazrs,laszip]


In [0]:
from pathlib import Path
from pyspark.sql import SparkSession
import laspy
import numpy as np
from typing import Generator, Optional, Tuple, Dict
import os

try:
    from pyspark.sql.datasource import DataSource, DataSourceReader
    from pyspark.sql.types import *
except ImportError as e:
    print(f"Error importing PySpark custom data sources: {e}. This feature is only available in Databricks Runtime 15.2 and above.")
    print("PySpark custom data sources are in Public Preview in Databricks Runtime 15.2 and above, and on serverless environment version 2. Streaming support is available in Databricks Runtime 15.3 and above.")
    raise

class LASToGeometryDataSourceReader(DataSourceReader):
    """
    Data source reader to read LAS/LAZ files and convert them to a dataframe.

    Attributes:
        schema (StructType): The schema of the output data, including fields for x, y, z, intensity and headers.
        options (dict): Configuration options to customize the data reader.

    Options:
        - `path` (str): The file path to the input LAS/LAZ file. **Required**.
        - `chunkSize` (int): Process records in chunks of `chunkSize` records at a time. Defaults to 1000000.

    Example Usage:
        df = (
            spark.read.format("las")
            .option("path", path)
            .option("chunkSize", "1000000")
            .load()
        )
    """
    def __init__(self, schema: StructType, options: dict):
        """
        Initialize the LASToGeometryDataSourceReader.

        Args:
            schema (StructType): The schema of the output data.
            options (dict): Options to configure the data reader, such as file path and filters.
        """
        self.schema: StructType = schema
        self.options: dict = options

    def check_directory(directory_path):
        if os.path.isdir(directory_path):
            print(f"Directory exists: {directory_path}")
        else:
            print(f"Directory does not exist: {directory_path}")

    def read(self, partition: Optional[int] = None) -> Generator[Tuple[int, str, Optional[str], Dict[str, str]], None, None]:
        """
        Read the LAS file and yield data for each points.

        Args:
            partition (Optional[int]): Partition index, if applicable. Not implemented.

        Yields:
            tuple: A tuple containing the point's data
        """
        # Extract options
        input_path: str = self.options.get("path")
        if not input_path:
            raise ValueError("The 'path' option is required.")

        chunk_size: int = self.options.get("chunkSize", 1000000)

        # TODO: process files in a directory (for now supports only file)

        with laspy.open(input_path) as f:

            # TODO: perform scaling here instead of using already scaled values

            for points in f.chunk_iterator(chunk_size):
                x_float = np.array(points.x).astype(float)
                y_float = np.array(points.y).astype(float)
                z_float = np.array(points.z).astype(float)
                
                # Check if the 'gps_time' field is present
                gps_time = points.gps_time if hasattr(points, 'gps_time') else [None] * len(x_float)
                red = points.red if hasattr(points, 'red') else [None] * len(x_float)
                green = points.green if hasattr(points, 'green') else [None] * len(x_float)
                blue = points.blue if hasattr(points, 'blue') else [None] * len(x_float)

                for point in zip(
                    x_float, y_float, z_float, points.intensity, points.return_number, points.number_of_returns,
                    points.scan_direction_flag, points.edge_of_flight_line, points.classification, points.synthetic,
                    points.key_point, points.withheld, points.scan_angle_rank, points.user_data, points.point_source_id,
                    gps_time, red, green, blue
                ):
                    yield point

class LASToGeometryDataSource(DataSource):
    """
    A custom data source to convert LAS/LAZ files to geometries in WKT, WKB, or GeoJSON format,
    including tags for each object, using MapType for tags.
    """
    @classmethod
    def name(cls) -> str:
        """
        Get the name of the data source.

        Returns:
            str: The name of the data source.
        """
        return "las"

    def schema(self) -> StructType:
        """
        Define the schema for the output data.
        The schema includes fields present in point format 3.

        Returns:
            StructType: The schema including these fields:
            - x: scaled x coordinate
            - y: scaled y coordinate
            - z: scaled z coordinate 
            - intensity: the integer representation of the pulse return magnitude
            - return_number: is the pulse return number for a given output pulse
            - number_of_returns: total number of returns for a given pulse
            - scan_direction_flag: the direction at which the scanner mirror was traveling at the time of the output pulse
            - edge_of_flight_line: data bit has a value of 1 only when the point is at the end of a scan. It is the last point on a given scan line before it changes direction.
            - classification: point classes
            - synthetic:
            - key_point:
            - withheld:
            - scan_angle_rank: valid range from -90 to +90. The Scan Angle Rank is the angle (rounded to the nearest integer in the absolute value sense) at which the laser point was output from the laser system including the roll of the aircraft.  
            - user_data: additional information about the point
            - point_source_id: This value indicates the file from which this point originated.
            - gps_time:  double floating point time tag value at which the point was acquired
            - red: The Red image channel value associated with this point 
            - green: The Green image channel value associated with this point 
            - blue: The Blue image channel value associated with this point 
        """
        return StructType([
            StructField("x", FloatType(), True),
            StructField("y", FloatType(), True),
            StructField("z", FloatType(), True),
            StructField("intensity", ShortType(), True),
            StructField("return_number", ShortType(), True),
            StructField("number_of_returns", ShortType(), True),
            StructField("scan_direction_flag", ByteType(), True),
            StructField("edge_of_flight_line", ByteType(), True),
            StructField("classification", ByteType(), True),
            StructField("synthetic", ByteType(), True),
            StructField("key_point", ByteType(), True),
            StructField("withheld", ByteType(), True),
            StructField("scan_angle_rank", ByteType(), True),
            StructField("user_data", ByteType(), True),
            StructField("point_source_id", ShortType(), True),
            StructField("gps_time", DoubleType(), True),
            StructField("red", ShortType(), True),
            StructField("green", ShortType(), True),
            StructField("blue", ShortType(), True)
        ])

    def reader(self, schema: StructType) -> LASToGeometryDataSourceReader:
        """
        Create a data source reader for reading the LAS file.

        Args:
            schema (StructType): The schema of the output data.

        Returns:
            LASToGeometryDataSourceReader: An instance of the data source reader.
        """
        return LASToGeometryDataSourceReader(schema, self.options)

def register_las_data_source():
    if os.getenv("IS_SERVERLESS") == "TRUE":
        raise RuntimeError(
            "Error: This data source can only be executed in a non-serverless context. "
            "Please attach the notebook to a traditional compute cluster and try again."
        )
    
    spark = SparkSession.getActiveSession()
    try:
        spark.dataSource.register(LASToGeometryDataSource)
        print("Custom data source 'las' registered successfully.")
    except AttributeError:
        print("Error registering custom data source: PySpark custom data sources are not supported in this environment.")


In [0]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from pyspark.sql import SparkSession
import laspy
import numpy as np
from typing import Generator, Optional, Tuple, Dict
import os

class LASToGeometryDataSourceReader(DataSourceReader):
    def __init__(self, schema: StructType, options: dict):
        self.schema = schema
        self.options = options

    def process_chunk(self, points):
        """Process a chunk of points."""
        processed_points = []
        for point in points:
            x_float = point.x.scaled_array()
            y_float = point.y.scaled_array()
            z_float = point.z.scaled_array()
            processed_points.append((x_float, y_float, z_float, point.intensity, {}))
        return processed_points

    def read(self, partition: Optional[int] = None) -> Generator[Tuple[int, str, Optional[str], Dict[str, str]], None, None]:
        input_path = self.options.get("path")
        if not input_path:
            raise ValueError("The 'path' option is required.")

        with laspy.open(input_path) as f:
            chunk_size = 1000000
            with ThreadPoolExecutor(max_workers=10) as executor:  # Adjust max_workers based on your system
                futures = []
                for points in f.chunk_iterator(chunk_size):
                    futures.append(executor.submit(self.process_chunk, points))

                for future in futures:
                    yield from future.result()

class LASToGeometryDataSource(DataSource):
    """
    A custom data source to convert LAS/LAZ files to geometries in WKT, WKB, or GeoJSON format,
    including tags for each object, using MapType for tags.
    """
    @classmethod
    def name(cls) -> str:
        """
        Get the name of the data source.

        Returns:
            str: The name of the data source.
        """
        return "las"

    def schema(self) -> StructType:
        """
        Define the schema for the output data.

        Returns:
            StructType: The schema including fields for ID, type, geometry, and tags.
            ['X',
 'Y',
 'Z',
 'intensity',
 'return_number',
 'number_of_returns',
 'scan_direction_flag',
 'edge_of_flight_line',
 'classification',
 'synthetic',
 'key_point',
 'withheld',
 'scan_angle_rank',
 'user_data',
 'point_source_id',
 'gps_time']
point format 0
        """
        return StructType([
            StructField("x", FloatType(), True),
            StructField("y", FloatType(), True),
            StructField("z", FloatType(), True),
            StructField("intensity", ShortType(), True),
            StructField("tags", MapType(StringType(), StringType()), True)
        ])

    def reader(self, schema: StructType) -> LASToGeometryDataSourceReader:
        """
        Create a data source reader for reading the PBF file.

        Args:
            schema (StructType): The schema of the output data.

        Returns:
            PBFToGeometryDataSourceReader: An instance of the data source reader.
        """
        return LASToGeometryDataSourceReader(schema, self.options)

def register_las_data_source():
    if os.getenv("IS_SERVERLESS") == "TRUE":
        raise RuntimeError(
            "Error: This data source can only be executed in a non-serverless context. "
            "Please attach the notebook to a traditional compute cluster and try again."
        )
    
    spark = SparkSession.getActiveSession()
    try:
        spark.dataSource.register(LASToGeometryDataSource)
        print("Custom data source 'las' registered successfully.")
    except AttributeError:
        print("Error registering custom data source: PySpark custom data sources are not supported in this environment.")


In [0]:
register_las_data_source()

In [0]:
input_path = '/Volumes/mpelletier/geospatial/lidar/Velodyne3_001.laz'

las = laspy.read(input_path)

for point in las:
    x_float = point.x.scaled_array()
    y_float = point.y.scaled_array()
    z_float = point.z.scaled_array()

    print (x_float, y_float, z_float, point.intensity, tags)


In [0]:
import numpy as np
import laspy

input_path = '/Volumes/mpelletier/geospatial/lidar/'

with laspy.open(input_path) as f:
    tags: Dict[str, str] = {}

    for points in f.chunk_iterator(10000):
        x_float = np.array(points.x).astype(float)
        y_float = np.array(points.y).astype(float)
        z_float = np.array(points.z).astype(float)
        
        for x, y, z in zip(x_float, y_float, z_float):
            print (x, y, z)

In [0]:
df = spark.read.format("las").option("path", "/Volumes/mpelletier/geospatial/lidar/Velodyne3_001.laz").load()
df.display()

In [0]:
from plotly.offline import init_notebook_mode, plot
import plotly.graph_objs as go

In [0]:
import plotly.graph_objs as go
from plotly.offline import init_notebook_mode, plot

init_notebook_mode(connected=True)

# Limit the DataFrame to 400,000 records
limited_df = df.limit(400000)

# Convert Spark DataFrame to Pandas DataFrame
pdf = limited_df.toPandas()

trace1 = go.Scatter3d(
  x=pdf['x'], y=pdf['y'], z=pdf['z'], mode='markers',  
  marker=dict(size=2, color=pdf['intensity'], colorscale='Viridis', opacity=1)
)

data = [trace1]
layout = go.Layout(
  autosize=True, width=1100, height=600,
  margin=dict(l=0, r=0, b=0, t=0),
  scene=dict(xaxis=dict(title="X"), yaxis=dict(title="Y"), zaxis=dict(title="Z"), aspectmode="data")
)

fig = go.Figure(data=data, layout=layout)
displayHTML(plot(fig, filename='3d-scatter-colorscale', output_type='div'))

In [0]:
from pyspark.sql.functions import col, min, max

# Calculate min and max for x, y, z
min_max_values = df.select(
    min(col("x")).alias("x_min"),
    max(col("x")).alias("x_max"),
    min(col("y")).alias("y_min"),
    max(col("y")).alias("y_max"),
    min(col("z")).alias("z_min"),
    max(col("z")).alias("z_max")
)

display(min_max_values)

In [0]:
df = df.cache()

In [0]:
df.write.saveAsTable("mpelletier.geospatial.lidar")

In [0]:
# z-order
# optimize / analyze

In [0]:
print(2150690-2149492.25)

In [0]:
print(1651079.375-1650248.875)

In [0]:
X_min = 2149492.25
Y_min = 1650248.875
h = 500
w = 200

lidar_3d_sdf = (
  spark.read.table("mpelletier.geospatial.lidar")
  .where(col("x").between(X_min, X_min + w - 1))
  .where(col("y").between(Y_min, Y_min + h - 1))
)

print(f"""count: {lidar_3d_sdf.count():,}""")

In [0]:
from pyspark.sql.functions import col

table_df = spark.read.table("mpelletier.geospatial.lidar")

# Define the fractions for each stratum
fractions = {0: 0.1, 1: 0.2, 2: 1.0}

# Apply sampleBy with the specified fractions
sampled_df = table_df.sample(False, 0.04, seed=42)

display(sampled_df.count())

In [0]:
import numpy as np

las = sampled_df.toPandas()

trace1 = go.Scatter3d(
  x=las.x, y=las.y, z=las.z, mode='markers',  
  marker=dict(size=2, color=las.intensity, colorscale='Viridis', opacity=1)
)

data = [trace1]
layout = go.Layout(
  autosize=True, width=1100, height=600,
  margin=dict(l=0, r=0, b=0, t=0),
  scene=dict(xaxis=dict(title="X"), yaxis=dict(title="Y"), zaxis=dict(title="Z"), aspectmode="data")
)

fig = go.Figure(data=data, layout=layout)
displayHTML(plot(fig, filename='3d-scatter-colorscale', output_type='div'))